# SimAMOC Analysis Notebook

Compare model output against observations. Load satellite data, visualize zonal profiles, diagnose biases.

**Data sources:**
- NOAA OLR CDR (outgoing longwave radiation)
- CERES net radiation & shortwave
- NASA NEO: cloud fraction, precipitation, albedo, snow cover
- NOAA OISST (sea surface temperature)
- WOA23 salinity

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import json, csv, os
from pathlib import Path

ROOT = Path('..').resolve()
DATA = ROOT / 'data'
EARTH = ROOT / 'earth-data'

plt.style.use('dark_background')
plt.rcParams['figure.figsize'] = (14, 4)
plt.rcParams['axes.facecolor'] = '#0a1018'
plt.rcParams['figure.facecolor'] = '#080e18'

def load_csv_grid(path, fill=99999.0):
    """Load a NEO CSV grid (360x180, no header). Returns masked array."""
    data = np.genfromtxt(path, delimiter=',')
    return np.ma.masked_values(data, fill)

def zonal_mean(grid):
    """Compute zonal mean (average over longitude)."""
    return np.ma.mean(grid, axis=1)

# Standard latitude arrays
lat_180 = np.linspace(89.5, -89.5, 180)   # NEO/CERES: north-first, 1° resolution
lat_model = np.linspace(-79.5, 79.5, 512)  # Model grid: south-first

print(f"Root: {ROOT}")
print(f"Earth data files: {sorted([f.name for f in EARTH.glob('*.csv')])}")
print(f"Model data files: {len(list(DATA.glob('*.json')))} JSON files")

## 1. Observed OLR — Zonal Profile

The model's energy balance determines everything. OLR (outgoing longwave radiation) is what the Earth emits to space. Key features to match:
- Subtropical peaks at ~20°N/S (clear skies)
- Equatorial dip (deep convective clouds trap OLR)
- Polar minimum (cold surface)

In [ ]:
with open(EARTH / 'olr_zonal_mean.json') as f:
    olr = json.load(f)

olr_lat = np.array(olr['lat'])
olr_vals = np.array(olr['zonal_mean_olr_wm2'])

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(olr_lat, olr_vals, color='#e05050', linewidth=2, label='NOAA OLR CDR')
ax.axhline(olr['global_mean_olr_wm2'], color='#e05050', alpha=0.3, linestyle='--', 
           label=f"Global mean: {olr['global_mean_olr_wm2']} W/m²")

# Model OLR for comparison (linearized: A_olr + B_olr * T)
# Using typical observed SST profile
sst_typical = 28 - 0.5 * np.abs(olr_lat) + 0.002 * olr_lat**2
A_olr, B_olr = 2.0, 0.1
# Scale model to W/m²: model uses ~7 units for ~240 W/m² peak solar
scale = 240 / 7.0
model_olr = (A_olr + B_olr * sst_typical) * scale
ax.plot(olr_lat, model_olr, color='#60c0e0', linewidth=1.5, linestyle='--', 
        label=f'Model OLR (A={A_olr}, B={B_olr}, scaled)')

ax.set_xlabel('Latitude')
ax.set_ylabel('OLR (W/m²)')
ax.set_title('Outgoing Longwave Radiation — Observed vs Model')
ax.legend(loc='lower center')
ax.set_xlim(-90, 90)
ax.set_ylim(100, 300)
ax.grid(alpha=0.15)
plt.tight_layout()
plt.show()

## 2. CERES Energy Balance — Where the Earth Gains and Loses Heat

Net radiation at TOA: positive = energy surplus (tropics), negative = energy deficit (poles).
This gradient is what drives poleward heat transport — the AMOC.

In [ ]:
net_rad = load_csv_grid(EARTH / 'net_radiation_ceres.csv')
sw_rad = load_csv_grid(EARTH / 'sw_radiation_ceres.csv')

zonal_net = zonal_mean(net_rad)
zonal_sw = zonal_mean(sw_rad)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Net radiation map
im0 = axes[0].imshow(net_rad, cmap='RdBu_r', vmin=-200, vmax=150, 
                      extent=[-180, 180, -90, 90], aspect='auto')
axes[0].set_title('Net Radiation at TOA (W/m²)')
plt.colorbar(im0, ax=axes[0], shrink=0.8)

# SW radiation map
im1 = axes[1].imshow(sw_rad, cmap='YlOrRd', vmin=0, vmax=45,
                      extent=[-180, 180, -90, 90], aspect='auto')
axes[1].set_title('Surface Shortwave (W/m²)')
plt.colorbar(im1, ax=axes[1], shrink=0.8)

# Zonal profiles
axes[2].plot(lat_180, zonal_net, color='#40c080', linewidth=2, label='Net radiation')
axes[2].plot(lat_180, zonal_sw, color='#e8a040', linewidth=1.5, label='Surface SW')
axes[2].axhline(0, color='white', alpha=0.2, linestyle='-')
axes[2].fill_between(lat_180, zonal_net, 0, where=zonal_net > 0, alpha=0.15, color='#e05050')
axes[2].fill_between(lat_180, zonal_net, 0, where=zonal_net < 0, alpha=0.15, color='#4080e0')
axes[2].set_xlabel('Latitude')
axes[2].set_ylabel('W/m²')
axes[2].set_title('Zonal Mean Energy Balance')
axes[2].legend()
axes[2].set_xlim(90, -90)
axes[2].grid(alpha=0.15)

plt.tight_layout()
plt.show()

# Key numbers
cos_lat = np.cos(np.radians(lat_180))
global_net = np.average(zonal_net, weights=cos_lat)
trop = np.mean(zonal_net[60:120])
polar = np.mean(np.concatenate([zonal_net[:30], zonal_net[150:]]))
print(f"Global mean net radiation: {global_net:.1f} W/m² (expect ~0.85)")
print(f"Tropical surplus (30N-30S): {trop:.1f} W/m²")
print(f"Polar deficit (60-90°): {polar:.1f} W/m²")
print(f"Equator-to-pole gradient: {trop - polar:.0f} W/m²")

## 3. Observed SST — What the Model Should Reproduce

Load the NOAA OISST data (already in model format) and show the target SST pattern.

In [ ]:
with open(DATA / 'sst.json') as f:
    sst_data = json.load(f)

sst = np.array(sst_data['sst']).reshape(512, 1024)
# Model grid: row 0 = north (lat=79.5), row 511 = south (lat=-79.5)
# Flip to south-first for consistency with model internals
sst_display = sst  # keep north-up for display

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# SST map
im = axes[0].imshow(sst_display, cmap='RdYlBu_r', vmin=-2, vmax=32,
                     extent=[-180, 180, -79.5, 79.5], aspect='auto')
axes[0].set_title('Observed SST (NOAA OISST)')
plt.colorbar(im, ax=axes[0], label='°C', shrink=0.8)

# Zonal mean SST
sst_masked = np.where(sst_display > -90, sst_display, np.nan)
zonal_sst = np.nanmean(sst_masked, axis=1)
lat_512 = np.linspace(79.5, -79.5, 512)

axes[1].plot(lat_512, zonal_sst, color='#e05050', linewidth=2)
axes[1].set_xlabel('Latitude')
axes[1].set_ylabel('SST (°C)')
axes[1].set_title('Zonal Mean SST')
axes[1].set_xlim(80, -80)
axes[1].grid(alpha=0.15)

# Reference points
for lat, label in [(0, 'Eq'), (30, '30N'), (-30, '30S'), (60, '60N')]:
    idx = np.argmin(np.abs(lat_512 - lat))
    axes[1].annotate(f'{label}: {zonal_sst[idx]:.1f}°C', 
                     xy=(lat, zonal_sst[idx]), fontsize=8, color='#8ab0c8')

plt.tight_layout()
plt.show()

## 4. All Observation Fields — Dashboard

Cloud fraction, precipitation, albedo, snow cover — the full picture of what drives climate.

In [ ]:
datasets = {
    'Cloud Fraction': ('cloud_fraction.csv', 'Blues_r', 0, 1),
    'Precipitation': ('precipitation.csv', 'YlGnBu', 0, 300),
    'Surface Albedo': ('albedo.csv', 'gray_r', 0, 0.8),
    'Snow Cover (%)': ('snow_cover.csv', 'cool', 0, 100),
    'Land Surface Temp': ('land_surface_temp.csv', 'RdYlBu_r', -30, 50),
    'NDVI (Vegetation)': ('ndvi.csv', 'YlGn', -0.1, 0.9),
}

fig, axes = plt.subplots(2, 3, figsize=(18, 8))
axes = axes.flatten()

for idx, (title, (fname, cmap, vmin, vmax)) in enumerate(datasets.items()):
    fpath = EARTH / fname
    if fpath.exists():
        grid = load_csv_grid(fpath)
        im = axes[idx].imshow(grid, cmap=cmap, vmin=vmin, vmax=vmax,
                              extent=[-180, 180, -90, 90], aspect='auto')
        axes[idx].set_title(title, fontsize=10)
        plt.colorbar(im, ax=axes[idx], shrink=0.7)
    else:
        axes[idx].text(0.5, 0.5, f'{fname}\nnot found', ha='center', va='center', 
                       transform=axes[idx].transAxes, color='#4a6a82')
        axes[idx].set_title(title, fontsize=10, color='#4a6a82')

plt.tight_layout()
plt.show()

## 5. Land Mask Inspection

Check where the mask might be thin — the user reported flows "through Africa."

In [ ]:
with open(DATA / 'mask.json') as f:
    mask_data = json.load(f)

# Decode hex mask (north-first, 1024x512)
bits = []
for c in mask_data['hex']:
    v = int(c, 16)
    bits.extend([(v >> 3) & 1, (v >> 2) & 1, (v >> 1) & 1, v & 1])

mask_nx = mask_data.get('nx', 1024)
mask_ny = mask_data.get('ny', 512)
mask_grid = np.array(bits[:mask_nx * mask_ny]).reshape(mask_ny, mask_nx)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Full mask
axes[0].imshow(mask_grid, cmap='ocean', extent=[-180, 180, -79.5, 79.5], aspect='auto')
axes[0].set_title(f'Land Mask ({mask_nx}x{mask_ny}) — blue=ocean, dark=land')

# Find thin land: count land width at each latitude (scan longitude)
# For each row, find runs of land cells and flag any < 3 cells wide
thin_land = np.zeros_like(mask_grid, dtype=float)
for j in range(mask_ny):
    row = mask_grid[j]
    in_land = False
    start = 0
    for i in range(mask_nx + 1):
        is_ocean = row[i % mask_nx] if i < mask_nx else True
        if not is_ocean and not in_land:
            start = i
            in_land = True
        elif is_ocean and in_land:
            width = i - start
            if width <= 3:
                thin_land[j, start:i] = 1.0
            in_land = False

axes[1].imshow(mask_grid, cmap='gray', extent=[-180, 180, -79.5, 79.5], aspect='auto', alpha=0.3)
axes[1].imshow(np.ma.masked_where(thin_land == 0, thin_land), cmap='Reds', 
               extent=[-180, 180, -79.5, 79.5], aspect='auto', vmin=0, vmax=1)
axes[1].set_title('Thin Land Features (<=3 cells wide) — potential leakage points')

plt.tight_layout()
plt.show()

# Count thin land pixels
print(f"Total thin land pixels: {int(thin_land.sum())} / {int((1-mask_grid).sum())} land pixels")
print(f"That's {thin_land.sum() / max(1, (1-mask_grid).sum()) * 100:.1f}% of all land")

## 6. Model Parameter Space

How do S_solar, A_olr, B_olr affect equilibrium SST? This shows the parameter sensitivity without running the sim.

In [ ]:
# Equilibrium SST: T_eq = (S_solar * cos(lat) - A_olr) / B_olr
lat = np.linspace(-80, 80, 200)
cos_lat = np.cos(np.radians(lat))

configs = [
    ('S=7, A=2, B=0.1 (baseline)', 7.0, 2.0, 0.1, '#4a9ec8'),
    ('S=8, A=2, B=0.05 (tuned)', 8.0, 2.0, 0.05, '#e8a040'),
    ('S=7, A=2, B=0.05', 7.0, 2.0, 0.05, '#40c080'),
    ('S=6, A=1.5, B=0.08', 6.0, 1.5, 0.08, '#e05050'),
]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for label, S, A, B, color in configs:
    T_eq = (S * cos_lat - A) / B
    axes[0].plot(lat, T_eq, color=color, linewidth=1.5, label=label)

# Overlay observed zonal SST
axes[0].plot(lat_512, zonal_sst, color='white', linewidth=2, alpha=0.5, label='Observed SST')
axes[0].set_xlabel('Latitude')
axes[0].set_ylabel('Equilibrium SST (°C)')
axes[0].set_title('Radiative Equilibrium Temperature')
axes[0].legend(fontsize=8)
axes[0].set_xlim(-80, 80)
axes[0].set_ylim(-10, 80)
axes[0].grid(alpha=0.15)

# B_olr sensitivity: how fast does temperature restore?
B_vals = np.linspace(0.02, 0.2, 50)
restoring_time = 1.0 / B_vals  # in model time units (~years)

axes[1].plot(B_vals, restoring_time, color='#4a9ec8', linewidth=2)
axes[1].set_xlabel('B_olr')
axes[1].set_ylabel('Restoring timescale (model years)')
axes[1].set_title('OLR Feedback Strength')
axes[1].axvline(0.1, color='#4a9ec8', alpha=0.3, linestyle='--', label='B=0.1 (baseline)')
axes[1].axvline(0.05, color='#e8a040', alpha=0.3, linestyle='--', label='B=0.05 (tuned)')
axes[1].legend(fontsize=8)
axes[1].grid(alpha=0.15)

plt.tight_layout()
plt.show()

---

## 7. Automated Model Capture

Run the sim headlessly with Puppeteer, extract SST + diagnostics, compare against observations.

**First time:** run this cell to capture the current production version. Takes ~30s.
After that, captured runs are cached in `notebooks/runs/` — re-running is instant.

In [ ]:
import subprocess, base64
from IPython.display import Image, display

RUNS = ROOT / 'notebooks' / 'runs'
RUNS.mkdir(exist_ok=True)

def capture(url, steps=25000):
    """Run sim headlessly and return the captured state dict."""
    hash_id = (url.split('amoc-')[1].split('-')[0]) if 'amoc-' in url else 'local'
    cache = RUNS / f"{hash_id}-{steps}.json"
    
    if cache.exists():
        print(f"Loading cached: {cache.name}")
        with open(cache) as f:
            return json.load(f)
    
    print(f"Capturing {hash_id} at {steps} steps...")
    result = subprocess.run(
        ['node', str(ROOT / 'scripts' / 'capture-sim.mjs'), url, str(steps), str(RUNS)],
        capture_output=True, text=True, timeout=120
    )
    print(result.stdout[-300:] if result.stdout else result.stderr[-300:])
    
    if cache.exists():
        with open(cache) as f:
            return json.load(f)
    return None

def load_all_runs():
    """Load all cached captures."""
    runs = {}
    for f in sorted(RUNS.glob('*.json')):
        with open(f) as fh:
            data = json.load(fh)
            label = f.stem  # e.g. "abc123-25000"
            runs[label] = data
    return runs

def show_screenshot(run_data, title=''):
    """Display a captured screenshot."""
    if 'screenshot' in run_data:
        img = base64.b64decode(run_data['screenshot'])
        display(Image(data=img, width=600))
        if title:
            print(title)

# Capture current production version
PROD_URL = 'https://amoc.vercel.app/v4-physics/'
current = capture(PROD_URL, steps=25000)

## 8. Model vs Observations — The Key Diagnostic

This is the plot that tells you if your changes helped. Shows:
- **SST bias map**: where is the model too warm (red) or too cold (blue)?
- **Zonal mean comparison**: model SST vs observed SST by latitude
- **RMSE**: single number to track progress (goal: < 3°C)

In [ ]:
def analyze_run(run_data, obs_sst_grid, label='Model'):
    """Compare a captured model run against observed SST. Returns fig."""
    if run_data is None:
        print("No run data — run the capture cell first")
        return
    
    # Model SST (360x180 downsampled, south-first)
    model_sst = np.array(run_data['sst']).reshape(run_data['sstShape'])
    model_lat = np.linspace(-79.5, 79.5, model_sst.shape[0])
    
    # Observed SST (512x1024, north-first) — downsample to match
    obs_ds = np.zeros((180, 360))
    for j in range(180):
        oj = int(j / 179 * 511)
        for i in range(360):
            oi = int(i / 359 * 1023)
            obs_ds[j, i] = obs_sst_grid[oj, oi]
    obs_ds_south = obs_ds[::-1]  # flip to south-first
    
    # Mask: only compare ocean cells
    ocean = (obs_ds_south > -90) & (model_sst != 0)
    bias = np.where(ocean, model_sst - obs_ds_south, np.nan)
    
    # RMSE
    valid = bias[~np.isnan(bias)]
    rmse = np.sqrt(np.mean(valid**2)) if len(valid) > 0 else float('nan')
    mean_bias = np.nanmean(valid) if len(valid) > 0 else float('nan')
    
    # Zonal means
    model_zonal = np.nanmean(np.where(ocean, model_sst, np.nan), axis=1)
    obs_zonal = np.nanmean(np.where(ocean, obs_ds_south, np.nan), axis=1)
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    # Model SST
    im0 = axes[0].imshow(model_sst[::-1], cmap='RdYlBu_r', vmin=-2, vmax=32,
                         extent=[-180, 180, -79.5, 79.5], aspect='auto')
    axes[0].set_title(f'{label} SST ({run_data["diagnostics"]["step"]} steps)')
    plt.colorbar(im0, ax=axes[0], label='°C', shrink=0.7)
    
    # Bias map
    im1 = axes[1].imshow(bias[::-1], cmap='RdBu_r', vmin=-10, vmax=10,
                         extent=[-180, 180, -79.5, 79.5], aspect='auto')
    axes[1].set_title(f'SST Bias (Model - Obs)  RMSE={rmse:.1f}°C')
    plt.colorbar(im1, ax=axes[1], label='°C bias', shrink=0.7)
    
    # Zonal comparison
    axes[2].plot(model_lat, obs_zonal, color='white', linewidth=2, alpha=0.6, label='Observed')
    axes[2].plot(model_lat, model_zonal, color='#e8a040', linewidth=2, label=label)
    axes[2].fill_between(model_lat, obs_zonal, model_zonal, alpha=0.15, color='#e05050')
    axes[2].set_xlabel('Latitude')
    axes[2].set_ylabel('SST (°C)')
    axes[2].set_title(f'Zonal Mean SST — RMSE {rmse:.2f}°C, bias {mean_bias:+.1f}°C')
    axes[2].legend()
    axes[2].set_xlim(-80, 80)
    axes[2].grid(alpha=0.15)
    
    plt.suptitle(label, fontsize=14, color='#7ab8de', y=1.02)
    plt.tight_layout()
    plt.show()
    
    # Print diagnostics
    d = run_data['diagnostics']
    print(f"  RMSE: {rmse:.2f}°C | Global SST: {d.get('globalSST', 0):.1f}°C | "
          f"AMOC: {d.get('amoc', 0):.3f} | Max vel: {d.get('maxVel', 0):.2f} | "
          f"KE: {d.get('KE', 0):.0f}")
    
    return {'rmse': rmse, 'mean_bias': mean_bias, 'diagnostics': d}

# Run analysis on current capture
if current:
    analyze_run(current, sst, label='Current Production')
else:
    print("No capture yet — run the cell above first")

## 9. Version Comparison — Did the Change Help?

Capture multiple versions and compare them side-by-side. Run the shell script first:
```bash
./scripts/capture-versions.sh 25000
```
Or capture specific versions below.

In [ ]:
# Load all cached runs and compare
runs = load_all_runs()

if len(runs) > 0:
    print(f"Found {len(runs)} cached runs:\n")
    
    # Summary table
    summaries = []
    for label, data in runs.items():
        d = data.get('diagnostics', {})
        summaries.append({
            'version': label,
            'steps': d.get('step', '?'),
            'rmse': f"{data.get('rmse', float('nan')):.2f}" if data.get('rmse') else 'N/A',
            'global_sst': f"{d.get('globalSST', 0):.1f}",
            'amoc': f"{d.get('amoc', 0):.3f}",
            'max_vel': f"{d.get('maxVel', 0):.2f}",
        })
    
    # Print as table
    print(f"{'Version':<25} {'Steps':>8} {'RMSE':>8} {'SST':>8} {'AMOC':>8} {'MaxV':>8}")
    print('-' * 75)
    for s in summaries:
        print(f"{s['version']:<25} {s['steps']:>8} {s['rmse']:>8} {s['global_sst']:>8} {s['amoc']:>8} {s['max_vel']:>8}")
    
    # RMSE comparison chart
    if len(runs) > 1:
        fig, ax = plt.subplots(figsize=(12, 4))
        labels = list(runs.keys())
        rmses = [runs[k].get('rmse', float('nan')) for k in labels]
        colors = ['#4a9ec8' if r == min([x for x in rmses if x]) else '#2a4a62' for r in rmses]
        ax.barh(range(len(labels)), rmses, color=colors)
        ax.set_yticks(range(len(labels)))
        ax.set_yticklabels([l[:30] for l in labels], fontsize=8)
        ax.set_xlabel('SST RMSE (°C)')
        ax.set_title('Version Comparison — Lower is Better')
        ax.axvline(3.0, color='#40c080', linestyle='--', alpha=0.5, label='Target: 3°C')
        ax.legend()
        ax.grid(alpha=0.15, axis='x')
        plt.tight_layout()
        plt.show()
else:
    print("No cached runs yet. Run this to capture versions:")
    print("  !cd .. && ./scripts/capture-versions.sh 25000")
    print("")
    print("Or capture one version manually:")
    print("  capture('https://amoc-xxx.vercel.app/v4-physics/', steps=25000)")